In [ ]:
# ============================================================
# FX-AlphaLab — Phase 2 + Phase 3
# Phase 2: Data Acquisition & Understanding (Correlation Agent)
# Phase 3: Modeling (Regression + Classification + BUY/SELL signals)
# ============================================================

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
import joblib


# ============================================================
# 0) CONFIG
# ============================================================
DATA_DIR = "/content"   # change if needed (ex: "." if local)
OUT_DIR  = "./outputs_fx_alphalab"
os.makedirs(OUT_DIR, exist_ok=True)

PAIR_FILES = {
    "EURUSD": "EURUSD_from_2018.csv",
    "GBPUSD": "GBPUSD_from_2018.csv",
    "USDJPY": "USDJPY_from_2018.csv",
    "EURJPY": "EURJPY_from_2018.csv",
    "USDCHF": "USDCHF_from_2018.csv",
    # optional:
    # "GBPJPY": "GBPJPY_from_2018.csv",
    # "EURGBP": "EURGBP_from_2018.csv",
}

# Phase 2 params
ROLL_WIN = 90
Z_TH     = 4.0
BREAK_TH = 0.30

# Phase 3 params
FEAT_WINS = [5, 10, 20, 50]
SPLIT_DATE = "2023-01-01"   # train < date, test >= date

# Signal thresholds
PROBA_BUY  = 0.55
PROBA_SELL = 0.45
MIN_EXPECTED_LOGRET = 0.0005


# ============================================================
# PHASE 2 — Data Acquisition & Understanding (Correlation Agent)
# ============================================================

print("\n" + "="*70)
print("PHASE 2 — DATA ACQUISITION & UNDERSTANDING")
print("="*70)

# 1) Load FX Close prices
prices = {}
for pair, fname in PAIR_FILES.items():
    fpath = os.path.join(DATA_DIR, fname)
    df = pd.read_csv(fpath, parse_dates=[0], index_col=0).sort_index()

    if "Close" not in df.columns:
        raise ValueError(f"'Close' column not found in {fname}. Please ensure file contains 'Close'.")

    prices[pair] = df["Close"].astype(float)

closes = pd.DataFrame(prices).sort_index()

print("\n=== Loaded series ===")
print("FX pairs:", list(closes.columns))
print("Date range:", closes.index.min(), "→", closes.index.max())

# 2) Data quality checks + alignment
print("\n=== Missing values per pair (raw) ===")
print(closes.isna().sum())
print("Total missing:", int(closes.isna().sum().sum()))

closes_aligned = closes.dropna().copy()

print("\n=== After alignment (dropna across all pairs) ===")
print("Rows:", len(closes_aligned), "| Columns:", closes_aligned.shape[1])
print("Any NA left?", closes_aligned.isna().any().any())

closes_aligned.to_csv(os.path.join(OUT_DIR, "closes_aligned.csv"))

# 3) Compute log-returns
returns = np.log(closes_aligned / closes_aligned.shift(1)).dropna()
returns.to_csv(os.path.join(OUT_DIR, "log_returns.csv"))

print("\n=== Returns quick stats (mean/std/min/max) ===")
print(returns.describe().T[["mean", "std", "min", "max"]])

# 4) Outlier detection on returns
print("\n=== Outliers on log-returns (|z| > %.1f) ===" % Z_TH)
outlier_report = []
for pair in returns.columns:
    series = returns[pair]
    z = (series - series.mean()) / series.std(ddof=0)
    outliers = series[np.abs(z) > Z_TH]
    outlier_report.append([pair, int(outliers.shape[0])])

    if outliers.empty:
        print(f"{pair}: 0 outliers")
    else:
        print(f"{pair}: {len(outliers)} outliers (showing last 3)")
        print(outliers.tail(3))

outlier_df = pd.DataFrame(outlier_report, columns=["Pair", "NumOutliers"])
outlier_df.to_csv(os.path.join(OUT_DIR, "outlier_report.csv"), index=False)

# 5) Plots
plt.figure(figsize=(12, 4))
closes_aligned[closes_aligned.columns[0]].plot()
plt.title(f"{closes_aligned.columns[0]} Close Price (Aligned)")
plt.xlabel("Date"); plt.ylabel("Close")
plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "example_close.png"))
plt.show()

plt.figure(figsize=(12, 5))
norm_prices = closes_aligned / closes_aligned.iloc[0]
for pair in norm_prices.columns:
    norm_prices[pair].plot(label=pair)
plt.title("Normalized Close Prices (start=1)")
plt.xlabel("Date"); plt.ylabel("Normalized")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "normalized_close_all.png"))
plt.show()

plt.figure(figsize=(12, 5))
sns.boxplot(data=returns)
plt.title("Daily Log-Returns Boxplot (All Pairs)")
plt.ylabel("Log-return")
plt.xticks(rotation=45)
plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "returns_boxplot.png"))
plt.show()

# 6) Correlation matrices
corr_pearson  = returns.corr(method="pearson")
corr_spearman = returns.corr(method="spearman")

corr_pearson.to_csv(os.path.join(OUT_DIR, "corr_matrix_pearson.csv"))
corr_spearman.to_csv(os.path.join(OUT_DIR, "corr_matrix_spearman.csv"))

print("\n=== Static correlation (Pearson on log-returns) ===")
print(corr_pearson.round(3))

plt.figure(figsize=(9, 7))
sns.heatmap(corr_pearson, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation Matrix (Pearson, log-returns)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "corr_heatmap_pearson.png"))
plt.show()

# 7) Rolling correlations examples
if "EURUSD" in returns.columns and "GBPUSD" in returns.columns:
    plt.figure(figsize=(12, 4))
    rc = returns["EURUSD"].rolling(ROLL_WIN).corr(returns["GBPUSD"])
    rc.plot()
    plt.title(f"Rolling Corr EURUSD vs GBPUSD ({ROLL_WIN}-day)")
    plt.xlabel("Date"); plt.ylabel("Correlation")
    plt.grid(True); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "rolling_corr_eurusd_gbpusd.png"))
    plt.show()

# 8) Stationarity check (ADF)
def adf_test(series, name="series"):
    series = series.dropna()
    stat, pvalue, *_ = adfuller(series, autolag="AIC")
    return {"name": name, "adf_stat": stat, "p_value": pvalue, "stationary_5pct": pvalue < 0.05}

adf_results = [adf_test(returns[col], name=col) for col in returns.columns]
adf_df = pd.DataFrame(adf_results).sort_values("p_value")
adf_df.to_csv(os.path.join(OUT_DIR, "adf_stationarity_returns.csv"), index=False)

print("\n=== ADF Stationarity Test (log-returns) ===")
print(adf_df)

print("\n✅ Phase 2 completed. Outputs saved in:", OUT_DIR)


# ============================================================
# PHASE 3 — MODELING (Regression + Classification + Signals)
# ============================================================

print("\n" + "="*70)
print("PHASE 3 — MODELING (REGRESSION + CLASSIFICATION + SIGNALS)")
print("="*70)

def rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0)
    down = (-delta).clip(lower=0)
    roll_up = up.rolling(window).mean()
    roll_down = down.rolling(window).mean()
    rs = roll_up / (roll_down + 1e-12)
    return 100 - (100 / (1 + rs))

def build_features_for_pair(pair: str, closes_df: pd.DataFrame, returns_df: pd.DataFrame) -> pd.DataFrame:
    px = closes_df[pair]
    r  = returns_df[pair]
    feat = pd.DataFrame(index=returns_df.index)

    # 1) Lags returns
    for lag in [1, 2, 3, 5, 10]:
        feat[f"ret_lag_{lag}"] = r.shift(lag)

    # 2) Rolling stats returns
    for w in FEAT_WINS:
        feat[f"ret_mean_{w}"] = r.rolling(w).mean()
        feat[f"ret_std_{w}"]  = r.rolling(w).std()

    # 3) Trend with moving averages
    for w in [10, 20, 50]:
        ma = px.rolling(w).mean()
        feat[f"ma_ratio_{w}"] = (px / (ma + 1e-12)) - 1.0

    # 4) RSI
    feat["rsi_14"] = rsi(px, 14)

    # 5) Regime proxy: average rolling correlation across all pairs
    rolling_corr_all = returns_df.rolling(ROLL_WIN).corr()
    rolling_corr_mean_by_date = rolling_corr_all.groupby(level=0).mean()
    feat["avg_roll_corr_all_pairs"] = rolling_corr_mean_by_date.mean(axis=1)

    # 6) Divergence vs top-3 correlated pairs (static)
    corr_p = returns_df.corr()
    top3 = corr_p[pair].drop(index=[pair]).sort_values(ascending=False).head(3).index.tolist()
    feat["divergence_vs_top3"] = np.abs(returns_df[pair] - returns_df[top3].mean(axis=1))

    # 7) Include other pairs returns as features (shifted)
    for other in returns_df.columns:
        if other != pair:
            feat[f"other_ret_{other}"] = returns_df[other].shift(1)

    return feat

def make_targets(pair: str, closes_df: pd.DataFrame):
    y_reg = closes_df[pair].shift(-1)  # Close(t+1)
    y_cls = (closes_df[pair].shift(-1) > closes_df[pair]).astype(int)  # Up/Down
    return y_reg, y_cls

def time_split(df: pd.DataFrame, split_date: str):
    train = df[df.index < split_date].copy()
    test  = df[df.index >= split_date].copy()
    return train, test

def regression_metrics(y_true, y_pred) -> dict:
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }

def classification_metrics(y_true, y_pred) -> dict:
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
    }

def build_signals(proba_up: np.ndarray, expected_logret: np.ndarray) -> np.ndarray:
    return np.where(
        (proba_up >= PROBA_BUY) & (expected_logret >= MIN_EXPECTED_LOGRET),
        "BUY",
        np.where(
            (proba_up <= PROBA_SELL) & (expected_logret <= -MIN_EXPECTED_LOGRET),
            "SELL",
            "HOLD"
        )
    )

# --- Training loop (1 model per pair) ---
results = []
conf_mats = {}

for pair in closes_aligned.columns:
    print(f"\n--- Modeling for {pair} ---")

    X = build_features_for_pair(pair, closes_aligned, returns)
    y_reg, y_cls = make_targets(pair, closes_aligned)

    data = X.copy()
    data["y_reg"] = y_reg
    data["y_cls"] = y_cls
    data = data.dropna()

    train, test = time_split(data, SPLIT_DATE)

    if len(train) < 300 or len(test) < 50:
        print(f"⚠️ Not enough rows for {pair}. Train={len(train)}, Test={len(test)}. Change SPLIT_DATE.")
        continue

    X_train = train.drop(columns=["y_reg", "y_cls"])
    X_test  = test.drop(columns=["y_reg", "y_cls"])

    yreg_train = train["y_reg"]
    yreg_test  = test["y_reg"]

    ycls_train = train["y_cls"]
    ycls_test  = test["y_cls"]

    # Regression model
    reg_model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])
    reg_model.fit(X_train, yreg_train)
    yreg_pred = reg_model.predict(X_test)
    reg_m = regression_metrics(yreg_test, yreg_pred)

    # Classification model
    cls_model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=2000))
    ])
    cls_model.fit(X_train, ycls_train)
    ycls_pred = cls_model.predict(X_test)
    proba_up = cls_model.predict_proba(X_test)[:, 1]
    cls_m = classification_metrics(ycls_test, ycls_pred)

    # Confusion matrix
    cm = confusion_matrix(ycls_test, ycls_pred)
    conf_mats[pair] = cm

    # Signals
    close_today = closes_aligned.loc[X_test.index, pair].values
    expected_logret = np.log((yreg_pred + 1e-12) / (close_today + 1e-12))
    signal = build_signals(proba_up, expected_logret)

    pred_df = pd.DataFrame({
        "Close_today": close_today,
        "Close_pred_t1": yreg_pred,
        "expected_logret": expected_logret,
        "proba_up": proba_up,
        "y_cls_true": ycls_test.values,
        "y_cls_pred": ycls_pred,
        "signal": signal
    }, index=X_test.index)

    # Save artifacts
    pred_df.to_csv(os.path.join(OUT_DIR, f"{pair}_predictions_signals.csv"))
    joblib.dump(reg_model, os.path.join(OUT_DIR, f"{pair}_reg_model.pkl"))
    joblib.dump(cls_model, os.path.join(OUT_DIR, f"{pair}_cls_model.pkl"))

    results.append({
        "Pair": pair,
        "TrainRows": int(len(train)),
        "TestRows": int(len(test)),
        "Reg_MAE": reg_m["MAE"],
        "Reg_RMSE": reg_m["RMSE"],
        "Reg_R2": reg_m["R2"],
        "Cls_Accuracy": cls_m["Accuracy"],
        "Cls_Precision": cls_m["Precision"],
        "Cls_Recall": cls_m["Recall"],
        "Cls_F1": cls_m["F1"],
    })

# --- CLEAR RESULTS DISPLAY (TABLE) ---
summary_df = pd.DataFrame(results).sort_values("Pair")
summary_path = os.path.join(OUT_DIR, "modeling_summary_metrics.csv")
summary_df.to_csv(summary_path, index=False)

print("\n" + "="*70)
print("✅ MODELING RESULTS SUMMARY (CLEAR TABLE)")
print("="*70)
print(summary_df.to_string(index=False))

# --- Plot: Accuracy comparison (clear chart) ---
plt.figure(figsize=(10, 4))
plt.bar(summary_df["Pair"], summary_df["Cls_Accuracy"])
plt.title("Classification Accuracy by Pair (UP/DOWN)")
plt.xlabel("Pair"); plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "accuracy_by_pair.png"))
plt.show()

# --- Plot confusion matrices (very clear) ---
for pair, cm in conf_mats.items():
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Confusion Matrix — {pair}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"confusion_matrix_{pair}.png"))
    plt.show()

print("\n✅ Phase 3 completed.")
print("Summary saved:", summary_path)
print("All outputs saved in:", OUT_DIR)

FileNotFoundError: [Errno 2] No such file or directory: '/content\\EURUSD_from_2018.csv'